In [1]:
!git clone https://github.com/Kaihua-Chen/diffusion-vas
%cd diffusion-vas

%mkdir checkpoints
%cd checkpoints
!git lfs install
!git clone https://huggingface.co/kaihuac/diffusion-vas-amodal-segmentation
!git clone https://huggingface.co/kaihuac/diffusion-vas-content-completion

!wget -O depth_anything_v2_vitl.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth?download=true
%cd ../..


Cloning into 'diffusion-vas'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 462 (delta 29), reused 28 (delta 28), pack-reused 427 (from 1)
Receiving objects: 100% (462/462), 102.79 MiB | 12.54 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/diffusion-vas
/content/diffusion-vas/checkpoints
Updated git hooks.
Git LFS initialized.
Cloning into 'diffusion-vas-amodal-segmentation'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 11), reused 0 (delta 0), pack-reused 4 (from 1)
Unpacking objects: 100% (48/48), 28.43 KiB | 2.58 MiB/s, done.
Filtering content: 100% (3/3), 3.03 GiB | 11.20 MiB/s, done.
Encountered 1 file(s) that may not have been copied correctly on Windows:
	unet/diffusion_pytorch_model.safetensors

See: `git lfs help smudge` for more details.
Cloning int

In [2]:
!tar -xvf ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz

Streaming output truncated to the last 5000 lines.
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00019.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00000.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00017.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00010.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00014.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00013.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00005.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00019.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00009.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00007.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00001.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00021.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00000.

In [3]:
from pathlib import Path
import shutil
import numpy as np
from PIL import Image

def copy_demo_format(
    root: Path,
    camera_index: int = 0,
    object_id: int = 1,
    target_root: Path = None
) -> Path:
    cam_folder = root / f"camera_{camera_index:04d}"

    if not cam_folder.exists():
        raise FileNotFoundError(f"Camera folder {cam_folder} not found.")

    if target_root is None:
        target_root = root

    # Create target structure
    new_folder = target_root / f"copy_cam_{camera_index}_obj_{object_id}"
    rgba_target = new_folder / "rgbs"
    seg_target = new_folder / "masks"
    rgba_target.mkdir(parents=True, exist_ok=True)
    seg_target.mkdir(parents=True, exist_ok=True)

    # Collect and sort source files
    rgba_files = sorted(cam_folder.glob("rgba_*.png"))
    seg_files = sorted(cam_folder.glob("segmentation_*.png"))

    assert len(rgba_files) == len(seg_files), "Mismatch between RGB and segmentation files."

    limit = 25
    count = min(limit, len(rgba_files))

    for i in range(count):
        # Copy RGBA image
        rgba_dest = rgba_target / f"rgba_{i:05d}.png"
        shutil.copy(rgba_files[i], rgba_dest)

        # Process and save mask for the correct object ID
        seg_img = np.array(Image.open(seg_files[i]))
        binary_mask = (seg_img == object_id).astype(np.uint8) * 255
        mask_img = Image.fromarray(binary_mask)
        mask_dest = seg_target / f"segmentation_{i:05d}.png"
        mask_img.save(mask_dest)

    # Pad RGBA and masks if fewer than 25 frames
    if count < limit:
        last_rgba = rgba_files[count - 1]
        last_seg = np.array(Image.open(seg_files[count - 1]))
        last_mask = (last_seg == object_id).astype(np.uint8) * 255
        last_mask_img = Image.fromarray(last_mask)

        for i in range(count, limit):
            shutil.copy(last_rgba, rgba_target / f"rgba_{i:05d}.png")
            last_mask_img.save(seg_target / f"segmentation_{i:05d}.png")

    return new_folder


In [4]:
root_path = Path("/content/ff5da6d6ecae486bb294aeaf5ee8f8a1")
target_path = Path("/content/diffusion-vas/demo_data")
copy_demo_format(root_path,camera_index=0,object_id=10,target_root=target_path)

PosixPath('/content/diffusion-vas/demo_data/copy_cam_0_obj_10')

In [5]:
%cd /content/diffusion-vas
!python demo.py --seq_name "copy_cam_0_obj_10"

/content/diffusion-vas
2025-07-24 16:44:53.143844: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 16:44:53.159773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753375493.180698    4761 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753375493.187199    4761 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-24 16:44:53.208597: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to 

In [9]:
import glob

amodal_dir = "/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010"
amodal_paths = sorted(glob.glob(f"{amodal_dir}/rgba_*.png"))

print(f"Found {len(amodal_paths)} PNGs.")
print(amodal_paths[:5])  # Preview a few

Found 24 PNGs.
['/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010/rgba_00000.png', '/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010/rgba_00001.png', '/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010/rgba_00002.png', '/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010/rgba_00003.png', '/content/ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0000/obj_0010/rgba_00004.png']


In [10]:
from PIL import Image

images = [Image.open(p).resize((256, 256)) for p in amodal_paths]
images[0].save("gt_amodal_rgb.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
print("GIF saved as gt_amodal_rgb.gif")

GIF saved as gt_amodal_rgb.gif


In [23]:
import imageio.v2 as imageio
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from PIL import Image

# Load file paths
gt_path = "/content/diffusion-vas/gt_amodal_rgb.gif"
pred_path = "/content/diffusion-vas/outputs/copy_cam_0_obj_10/pred_amodal_rgb.gif"

gt_frames = imageio.mimread(gt_path)
pred_frames = imageio.mimread(pred_path)

# Align lengths
min_len = min(len(gt_frames), len(pred_frames))
gt_frames = gt_frames[:min_len]
pred_frames = pred_frames[:min_len]

# Resize and convert to RGB
target_size = (256, 256)
gt_np = np.stack([
    np.array(Image.fromarray(f).resize(target_size).convert("RGB")).astype(np.float32) / 255.0
    for f in gt_frames
])
pred_np = np.stack([
    np.array(Image.fromarray(f).resize(target_size).convert("RGB")).astype(np.float32) / 255.0
    for f in pred_frames
])

# === Normalize background to white in both GT and prediction ===
def normalize_background_to_white(x):
    bg_mask = (x < 0.05).all(axis=-1, keepdims=True)
    return np.where(bg_mask, 1.0, x)

gt_np = np.stack([normalize_background_to_white(f) for f in gt_np])
pred_np = np.stack([normalize_background_to_white(f) for f in pred_np])

# === Compute metrics ===
psnr_list, ssim_list, ace_list = [], [], []

for gt, pr in zip(gt_np, pred_np):
    psnr_list.append(psnr(gt, pr, data_range=1.0))
    ssim_list.append(ssim(gt, pr, channel_axis=2, data_range=1.0))

    # Object-only ACE: ignore pixels where both are white (background)
    background_mask = ((gt > 0.95) & (pr > 0.95)).all(axis=-1)
    object_mask = ~background_mask

    gt_obj = gt[object_mask]
    pr_obj = pr[object_mask]

    if len(gt_obj) > 0:
        ace = np.mean(np.abs(gt_obj - pr_obj))
        ace_list.append(ace)

# Final results
results = {
    "Average PSNR": np.mean(psnr_list),
    "Average SSIM": np.mean(ssim_list),
    "ACE (Object Pixels Only, White BG)": np.mean(ace_list)
}

results



{'Average PSNR': np.float64(29.78410205201784),
 'Average SSIM': np.float32(0.9857257),
 'ACE (Object Pixels Only, White BG)': np.float32(0.13768393)}

In [ ]:
def compute_iou(pred, true):
    intersection = (pred & true).sum()
    union = (pred | true).sum()
    return intersection / union if union != 0 else 0

In [ ]:
from skimage.metrics import structural_similarity as ssim

def compute_ssim(pred, true):
    return ssim(pred, true, channel_axis=-1, data_range=1.0)


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr

def compute_psnr(pred, true):
    return psnr(true, pred, data_range=1.0)


In [ ]:
import numpy as np

def compute_ace(pred, true):
    return np.mean(np.abs(pred.astype(np.float32) - true.astype(np.float32)))
